In [1]:

import sys
import os
import numpy as np
import pandas as pd
import time
import pickle

from sklearn.linear_model import LogisticRegression

sys.path.append(os.path.join(os.getcwd(), '..', '..'))
from src.preprocessing import apply_smote
from src.evaluation import compute_metrics, save_results

RANDOM_STATE = 42

In [ ]:
DATA_DIR = "../../data/processed"

X_train_full = pd.read_pickle(f"{DATA_DIR}/X_train_scaled.pkl")
X_test_full = pd.read_pickle(f"{DATA_DIR}/X_test_scaled.pkl")
y_train = pd.read_pickle(f"{DATA_DIR}/y_train.pkl")
y_test = pd.read_pickle(f"{DATA_DIR}/y_test.pkl")
target_encoder = pd.read_pickle(f"{DATA_DIR}/target_encoder.pkl")
target_names = list(target_encoder.classes_)

# Convert to numpy if needed
if isinstance(y_train, pd.Series):
    y_train = y_train.values
if isinstance(y_test, pd.Series):
    y_test = y_test.values

# Feature subsets
X_train_top10 = pd.read_pickle(f"{DATA_DIR}/X_train_top10.pkl")
X_test_top10 = pd.read_pickle(f"{DATA_DIR}/X_test_top10.pkl")
X_train_top20 = pd.read_pickle(f"{DATA_DIR}/X_train_top20.pkl")
X_test_top20 = pd.read_pickle(f"{DATA_DIR}/X_test_top20.pkl")
X_train_top30 = pd.read_pickle(f"{DATA_DIR}/X_train_top30.pkl")
X_test_top30 = pd.read_pickle(f"{DATA_DIR}/X_test_top30.pkl")
X_train_pca = pd.read_pickle(f"{DATA_DIR}/X_train_pca.pkl")
X_test_pca = pd.read_pickle(f"{DATA_DIR}/X_test_pca.pkl")

print(f"Data loaded. Classes: {len(target_names)}")
print(f"Feature sets: full={X_train_full.shape[1]}, top10={X_train_top10.shape[1]}, "
      f"top20={X_train_top20.shape[1]}, top30={X_train_top30.shape[1]}, pca={X_train_pca.shape[1]}")



Data loaded. Classes: 12
Feature sets: full=83, top10=10, top20=20, top30=30, pca=26


In [ ]:
def run_lr_experiment(X_train, y_train, X_test, y_test,
                      feature_set_name, imbalance_strategy,
                      target_encoder=None):
    """
    Run one Logistic Regression experiment.
    """
    model_name = f"LR | {feature_set_name} | {imbalance_strategy}"
    print(f"# {model_name}")
    

    # Apply imbalance strategy
    if imbalance_strategy == "smote":
        print("  Applying SMOTE...")
        X_tr, y_tr = apply_smote(X_train, y_train)
    else:
        X_tr, y_tr = X_train, y_train

    # Create model
    if imbalance_strategy == "cost-sensitive":
        model = LogisticRegression(
            class_weight='balanced',
            max_iter=1000,
            random_state=RANDOM_STATE
        )
    else:
        model = LogisticRegression(
            max_iter=1000,
            random_state=RANDOM_STATE
        )

    # Train
    start_time = time.time()
    print("  Training...")
    model.fit(X_tr, y_tr)
    train_time = time.time() - start_time
    print(f"  Training time: {train_time:.1f}s")

    # Predict
    y_pred = model.predict(X_test)

    # Evaluate using shared evaluation.py
    metrics = compute_metrics(y_test, y_pred, target_encoder=target_encoder)
    metrics["feature_set"] = feature_set_name
    metrics["imbalance_strategy"] = imbalance_strategy
    metrics["training_time"] = train_time

    # Save results
    exp_name = f"lr_{feature_set_name}_{imbalance_strategy}"
    save_results(metrics, exp_name, save_dir="../../results")

    return metrics

In [4]:
feature_sets = {
    "full-83": (X_train_full, X_test_full),
    "top-30": (X_train_top30, X_test_top30),
    "top-20": (X_train_top20, X_test_top20),
    "top-10": (X_train_top10, X_test_top10),
    "pca": (X_train_pca, X_test_pca),
}

imbalance_strategies = ["baseline", "smote", "cost-sensitive"]

all_results = {}

for fs_name, (X_tr, X_te) in feature_sets.items():
    for strategy in imbalance_strategies:
        exp_name = f"LR | {fs_name} | {strategy}"
        result = run_lr_experiment(
            X_train=X_tr,
            y_train=y_train,
            X_test=X_te,
            y_test=y_test,
            feature_set_name=fs_name,
            imbalance_strategy=strategy,
            target_encoder=target_encoder,
        )
        all_results[exp_name] = result

print("\n\nAll 15 LR experiments complete!")


############################################################
# LR | full-83 | baseline
############################################################
  Training...


/Users/mahip/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/mahip/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/mahip/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/mahip/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:336: RuntimeWarning: divide by zero encountered in matmul
  grad[:, :n_features] = grad_pointwise.T @ X + l2_reg_strength * weights
/Users/mahip/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:

  Training time: 3.2s

  Macro F1 (PRIMARY):  0.9179
  Accuracy:            0.9911
  Macro Precision:     0.9352
  Macro Recall:        0.9090

  Saved report to ../../results/lr_full-83_baseline_report.csv
  Saved summary to ../../results/lr_full-83_baseline_summary.csv

############################################################
# LR | full-83 | smote
############################################################
  Applying SMOTE...
  Before SMOTE: 98,493 samples


/Users/mahip/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/mahip/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/mahip/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/mahip/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


  After SMOTE:  908,724 samples
  Training...


...

/Users/mahip/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/mahip/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/mahip/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/mahip/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


  After SMOTE:  908,724 samples
  Training...


/Users/mahip/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/mahip/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/mahip/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/mahip/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:336: RuntimeWarning: divide by zero encountered in matmul
  grad[:, :n_features] = grad_pointwise.T @ X + l2_reg_strength * weights
/Users/mahip/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:

  Training time: 61.5s

  Macro F1 (PRIMARY):  0.8092
  Accuracy:            0.9822
  Macro Precision:     0.7716
  Macro Recall:        0.9463

  Saved report to ../../results/lr_pca_smote_report.csv
  Saved summary to ../../results/lr_pca_smote_summary.csv

############################################################
# LR | pca | cost-sensitive
############################################################
  Training...
  Training time: 7.0s

  Macro F1 (PRIMARY):  0.7877
  Accuracy:            0.9797
  Macro Precision:     0.7534
  Macro Recall:        0.9352

  Saved report to ../../results/lr_pca_cost-sensitive_report.csv
  Saved summary to ../../results/lr_pca_cost-sensitive_summary.csv


All 15 LR experiments complete!


/Users/mahip/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/mahip/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/mahip/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/mahip/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [5]:
from src.evaluation import compare_experiments
compare_experiments(all_results)


  EXPERIMENT COMPARISON (sorted by Macro F1)
                               Macro F1  Accuracy  Macro Precision  Macro Recall
Experiment                                                                      
LR | full-83 | baseline          0.9179    0.9911           0.9352        0.9090
LR | pca | baseline              0.8568    0.9809           0.9291        0.8249
LR | top-30 | baseline           0.8544    0.9858           0.8869        0.8329
LR | top-20 | baseline           0.8471    0.9870           0.8760        0.8273
LR | full-83 | smote             0.8370    0.9889           0.8042        0.9568
LR | top-20 | smote              0.8227    0.9789           0.7933        0.9444
LR | full-83 | cost-sensitive    0.8197    0.9874           0.7850        0.9595
LR | pca | smote                 0.8092    0.9822           0.7716        0.9463
LR | top-20 | cost-sensitive     0.8073    0.9826           0.7711        0.9524
LR | top-30 | smote              0.8026    0.9795           0.7

,Macro F1,Accuracy,Macro Precision,Macro Recall
Experiment,,,,
LR | full-83 | baseline,0.917922,0.991066,0.935213,0.908956
LR | pca | baseline,0.856803,0.980913,0.929091,0.824865
LR | top-30 | baseline,0.854403,0.985827,0.886873,0.832945
LR | top-20 | baseline,0.847102,0.987005,0.875951,0.827319
LR | full-83 | smote,0.837012,0.988913,0.804245,0.956829
LR | top-20 | smote,0.822726,0.978882,0.793271,0.944424
LR | full-83 | cost-sensitive,0.819710,0.987411,0.785003,0.959513
LR | pca | smote,0.809219,0.982172,0.771634,0.946283
LR | top-20 | cost-sensitive,0.807275,0.982578,0.771121,0.952400


In [6]:
results_dir = "../../results"
for f in sorted(os.listdir(results_dir)):
    if f.startswith("lr_") and f.endswith("_report.csv"):
        name = f.replace("_report.csv", "")
        df = pd.read_csv(os.path.join(results_dir, f), index_col=0)
        # Find Metasploit and NMAP_FIN_SCAN rows
        metasploit = df.loc["Metasploit_Brute_Force_SSH"] if "Metasploit_Brute_Force_SSH" in df.index else df.iloc[4]
        nmap_fin = df.loc["NMAP_FIN_SCAN"] if "NMAP_FIN_SCAN" in df.index else df.iloc[5]
        print(f"\n{name}")
        print(f"  Metasploit_Brute_Force_SSH: precision={metasploit['precision']:.3f}  recall={metasploit['recall']:.3f}  f1={metasploit['f1-score']:.3f}")
        print(f"  NMAP_FIN_SCAN:              precision={nmap_fin['precision']:.3f}  recall={nmap_fin['recall']:.3f}  f1={nmap_fin['f1-score']:.3f}")



lr_full-83_baseline
  Metasploit_Brute_Force_SSH: precision=0.857  recall=0.857  f1=0.857
  NMAP_FIN_SCAN:              precision=0.714  recall=0.833  f1=0.769

lr_full-83_cost-sensitive
  Metasploit_Brute_Force_SSH: precision=0.127  recall=1.000  f1=0.226
  NMAP_FIN_SCAN:              precision=0.185  recall=0.833  f1=0.303

lr_full-83_smote
  Metasploit_Brute_Force_SSH: precision=0.103  recall=1.000  f1=0.187
  NMAP_FIN_SCAN:              precision=0.333  recall=0.833  f1=0.476

lr_pca_baseline
  Metasploit_Brute_Force_SSH: precision=1.000  recall=0.429  f1=0.600
  NMAP_FIN_SCAN:              precision=1.000  recall=0.833  f1=0.909

lr_pca_cost-sensitive
  Metasploit_Brute_Force_SSH: precision=0.121  recall=1.000  f1=0.215
  NMAP_FIN_SCAN:              precision=0.179  recall=0.833  f1=0.294

lr_pca_smote
  Metasploit_Brute_Force_SSH: precision=0.111  recall=1.000  f1=0.200
  NMAP_FIN_SCAN:              precision=0.333  recall=0.833  f1=0.476

lr_top-10_baseline
  Metasploit_Brute_F